# Smart Expense Classifier - Arabic Data ML Pipeline

**Dataset:** Arabic Transaction Dataset (10K Egyptian Arabic records)

**Pipeline Steps:**
1. Load Arabic Dataset
2. Exploratory Data Analysis (EDA)
3. Arabic Text Preprocessing (with Arabic-specific cleaning)
4. Feature Extraction (TF-IDF with Arabic configuration)
5. Train-Test Split
6. Baseline Model Training
7. Model Comparison (Multiple Algorithms)
8. Hyperparameter Tuning
9. Model Evaluation & Metrics
10. Save Final Model

---

## 1. Setup & Imports

In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
from collections import Counter
import joblib
import time
import json
from datetime import datetime

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Arabic text processing
import arabic_reshaper
from bidi.algorithm import get_display

# Settings
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Configure matplotlib for Arabic text display
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(" Setup complete!")
print(f" Pipeline started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

 Setup complete!
 Pipeline started: 2026-11-16 13:36:17


## 2. Load Arabic Dataset

In [ ]:
# Load the generated Arabic dataset
print(" Loading Arabic transaction dataset...")
start_time = time.time()

# Try different possible locations
data_paths = [
    '../data/processed/arabic_only_dataset.csv',
    '../data/raw/arabic_only_dataset.csv',
    '../arabic_only_dataset.csv'
]

df = None
for path in data_paths:
    try:
        df = pd.read_csv(path, encoding='utf-8')
        print(f" Dataset loaded from: {path}")
        break
    except:
        continue

if df is None:
    print(" Dataset not found! Please ensure arabic_only_dataset.csv exists.")
else:
    load_time = time.time() - start_time
    print(f"⏱ Loading time: {load_time:.2f} seconds")
    print(f" Dataset shape: {df.shape}")
    print(f" Columns: {list(df.columns)}")

 Loading Arabic transaction dataset...
 Dataset not found! Please ensure arabic_only_dataset.csv exists.


In [3]:
# Display first few rows
print("\n First 5 records:")
df.head()


 First 5 records:


AttributeError: 'NoneType' object has no attribute 'head'

In [ ]:
# Dataset info
print("\n Dataset Information:")
df.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print(" Dataset Statistics:")
print(f"Total records: {len(df):,}")
print(f"Total categories: {df['category'].nunique()}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate records: {df.duplicated().sum()}")

In [ ]:
# Category distribution
print("\n Category Distribution:")
category_counts = df['category'].value_counts()
print(category_counts)
print(f"\nMost common: {category_counts.index[0]} ({category_counts.iloc[0]} records)")
print(f"Least common: {category_counts.index[-1]} ({category_counts.iloc[-1]} records)")

In [ ]:
# Visualize category distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
category_counts.plot(kind='bar', ax=axes[0], color='#3B82F6', edgecolor='black')
axes[0].set_title('Category Distribution (Bar Chart)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Pie chart
colors = sns.color_palette('husl', len(category_counts))
axes[1].pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
            startangle=90, colors=colors)
axes[1].set_title('Category Distribution (Pie Chart)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Transaction text length analysis
df['text_length'] = df['transaction'].str.len()
df['word_count'] = df['transaction'].str.split().str.len()

print("\n Text Length Statistics:")
print(df[['text_length', 'word_count']].describe())

In [ ]:
# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Character length
axes[0].hist(df['text_length'], bins=50, color='#06B6D4', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Transaction Text Length (Characters)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character Count', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["text_length"].mean():.1f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Word count
axes[1].hist(df['word_count'], bins=30, color='#10B981', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Word Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["word_count"].mean():.1f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sample transactions from each category
print("\n Sample Transactions by Category:")
for category in df['category'].unique():
    samples = df[df['category'] == category]['transaction'].head(3).tolist()
    print(f"\n{category}:")
    for i, sample in enumerate(samples, 1):
        print(f"  {i}. {sample}")

## 4. Arabic Text Preprocessing

In [ ]:
def preprocess_arabic_text(text):
    """
    Preprocess Arabic transaction text
    
    Steps:
    1. Convert to lowercase (for consistency)
    2. Remove Arabic diacritics (تشكيل)
    3. Remove English letters and numbers
    4. Remove punctuation
    5. Remove extra whitespace
    6. Keep only Arabic letters and spaces
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Remove Arabic diacritics (harakat)
    arabic_diacritics = re.compile(r'[\u064B-\u065F\u0670]')
    text = arabic_diacritics.sub('', text)
    
    # Remove English letters
    text = re.sub(r'[a-zA-Z]', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation and special characters (keep only Arabic letters and spaces)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    
    # Normalize Arabic letters
    text = text.replace('أ', 'ا')
    text = text.replace('إ', 'ا')
    text = text.replace('آ', 'ا')
    text = text.replace('ة', 'ه')
    text = text.replace('ى', 'ي')
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text.strip()

# Test the preprocessing function
sample_text = df['transaction'].iloc[0]
print(f"Original: {sample_text}")
print(f"Processed: {preprocess_arabic_text(sample_text)}")

In [ ]:
# Apply preprocessing to all transactions
print("\n Preprocessing Arabic text...")
start_time = time.time()

df['processed_text'] = df['transaction'].apply(preprocess_arabic_text)

# Remove empty texts after preprocessing
empty_count = (df['processed_text'] == '').sum()
if empty_count > 0:
    print(f" Found {empty_count} empty transactions after preprocessing. Removing...")
    df = df[df['processed_text'] != '']

processing_time = time.time() - start_time
print(f" Preprocessing complete!")
print(f"⏱ Processing time: {processing_time:.2f} seconds")
print(f" Final dataset shape: {df.shape}")

In [ ]:
# Compare original vs processed text
print("\n Original vs Processed Text (Sample):")
comparison_df = df[['transaction', 'processed_text', 'category']].head(10)
comparison_df

## 5. Train-Test Split

In [ ]:
# Prepare features and target
X = df['processed_text']
y = df['category']

print(f" Features (X) shape: {X.shape}")
print(f" Target (y) shape: {y.shape}")
print(f"\n Target classes: {sorted(y.unique())}")

In [ ]:
# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y  # Maintain category distribution
)

print("\n Train-Test Split:")
print(f"Training set: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set: {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")

print("\n Category distribution in splits:")
print("\nTraining:")
print(y_train.value_counts())
print("\nTesting:")
print(y_test.value_counts())

## 6. Feature Extraction (TF-IDF)

In [ ]:
# Initialize TF-IDF Vectorizer (optimized for Arabic)
print(" Initializing TF-IDF Vectorizer...")

tfidf = TfidfVectorizer(
    max_features=5000,           # Top 5000 features
    ngram_range=(1, 2),          # Unigrams and bigrams
    min_df=2,                     # Ignore terms appearing in less than 2 documents
    max_df=0.8,                   # Ignore terms appearing in more than 80% of documents
    sublinear_tf=True,           # Use logarithmic form for term frequency
    strip_accents='unicode',     # Remove accents
    analyzer='word',             # Analyze at word level
    token_pattern=r'\b[\u0600-\u06FF]+\b'  # Arabic unicode pattern
)

print(" Vectorizer initialized!")

In [ ]:
# Fit and transform training data
print("\n Transforming training data...")
start_time = time.time()

X_train_tfidf = tfidf.fit_transform(X_train)

transform_time = time.time() - start_time
print(f" Training data transformed!")
print(f"⏱ Transform time: {transform_time:.2f} seconds")
print(f" TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f" Vocabulary size: {len(tfidf.vocabulary_)}")

In [ ]:
# Transform test data
print("\n Transforming test data...")
X_test_tfidf = tfidf.transform(X_test)
print(f" Test data transformed!")
print(f" Test TF-IDF matrix shape: {X_test_tfidf.shape}")

In [ ]:
# Analyze most important features (words)
print("\n Top 20 Most Important Features (by average TF-IDF):")
feature_names = tfidf.get_feature_names_out()
tfidf_means = X_train_tfidf.mean(axis=0).A1
top_features_idx = tfidf_means.argsort()[-20:][::-1]

for i, idx in enumerate(top_features_idx, 1):
    print(f"{i:2d}. {feature_names[idx]:20s} (TF-IDF: {tfidf_means[idx]:.4f})")

## 7. Baseline Model - Logistic Regression

In [ ]:
# Train baseline model
print(" Training Baseline Model (Logistic Regression)...")
start_time = time.time()

baseline_model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    n_jobs=-1,  # Use all CPU cores
    verbose=1
)

baseline_model.fit(X_train_tfidf, y_train)

train_time = time.time() - start_time
print(f"\n Baseline model trained!")
print(f"⏱ Training time: {train_time:.2f} seconds")

In [ ]:
# Evaluate baseline model
print("\n Baseline Model Evaluation:")

# Training set performance
y_train_pred = baseline_model.predict(X_train_tfidf)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy*100:.2f}%")

# Test set performance
y_test_pred = baseline_model.predict(X_test_tfidf)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# Additional metrics
precision = precision_score(y_test, y_test_pred, average='weighted')
recall = recall_score(y_test, y_test_pred, average='weighted')
f1 = f1_score(y_test, y_test_pred, average='weighted')

print(f"\nPrecision: {precision*100:.2f}%")
print(f"Recall: {recall*100:.2f}%")
print(f"F1-Score: {f1*100:.2f}%")

In [ ]:
# Classification report
print("\n Detailed Classification Report:")
print(classification_report(y_test, y_test_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(y.unique()), 
            yticklabels=sorted(y.unique()),
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Baseline Model', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Category', fontsize=12)
plt.ylabel('True Category', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Model Comparison

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100, n_jobs=-1)
}

print(" Training and comparing multiple models...\n")

In [ ]:
# Train and evaluate each model
results = []

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f" Training: {name}")
    print('='*60)
    
    start_time = time.time()
    
    # Train
    model.fit(X_train_tfidf, y_train)
    
    # Predict
    y_pred = model.predict(X_test_tfidf)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    train_time = time.time() - start_time
    
    # Store results
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Training Time (s)': train_time
    })
    
    print(f" Completed in {train_time:.2f}s")
    print(f"Accuracy: {accuracy*100:.2f}%")
    print(f"F1-Score: {f1*100:.2f}%")

print(f"\n{'='*60}")
print(" All models trained!")
print('='*60)

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False)

print("\n Model Comparison Results:")
print(results_df.to_string(index=False))

print(f"\n Best Model: {results_df.iloc[0]['Model']}")
print(f"   Accuracy: {results_df.iloc[0]['Accuracy']*100:.2f}%")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors = ['#3B82F6', '#06B6D4', '#10B981', '#8B5CF6']

for idx, (ax, metric, color) in enumerate(zip(axes.flat, metrics, colors)):
    data = results_df.sort_values(metric, ascending=True)
    ax.barh(data['Model'], data[metric], color=color, edgecolor='black', alpha=0.8)
    ax.set_xlabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'Model Comparison - {metric}', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(data[metric]):
        ax.text(v + 0.01, i, f'{v*100:.2f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Hyperparameter Tuning (Best Model)

In [ ]:
# Get best model from comparison
best_model_name = results_df.iloc[0]['Model']
print(f" Tuning hyperparameters for: {best_model_name}\n")

# Define parameter grid based on best model
if 'Logistic' in best_model_name:
    param_grid = {
        'C': [0.1, 1, 10],
        'solver': ['lbfgs', 'liblinear'],
        'penalty': ['l2']
    }
    base_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, n_jobs=-1)
    
elif 'Naive Bayes' in best_model_name:
    param_grid = {
        'alpha': [0.1, 0.5, 1.0, 2.0]
    }
    base_model = MultinomialNB()
    
elif 'SVM' in best_model_name:
    param_grid = {
        'C': [0.1, 1, 10],
        'loss': ['hinge', 'squared_hinge']
    }
    base_model = LinearSVC(random_state=RANDOM_STATE, max_iter=1000)
    
else:  # Random Forest
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5]
    }
    base_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

print(f"Parameter grid: {param_grid}")

In [ ]:
# Perform GridSearchCV
print("\n Starting GridSearchCV...")
start_time = time.time()

grid_search = GridSearchCV(
    base_model,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_tfidf, y_train)

search_time = time.time() - start_time
print(f"\n GridSearchCV completed!")
print(f"⏱ Total search time: {search_time:.2f} seconds")

In [ ]:
# Best parameters and score
print("\n Best Parameters:")
print(grid_search.best_params_)

print(f"\n Best Cross-Validation Score: {grid_search.best_score_*100:.2f}%")

# Get best model
best_tuned_model = grid_search.best_estimator_

In [ ]:
# Evaluate tuned model on test set
print("\n Tuned Model Test Performance:")

y_test_pred_tuned = best_tuned_model.predict(X_test_tfidf)

test_accuracy_tuned = accuracy_score(y_test, y_test_pred_tuned)
precision_tuned = precision_score(y_test, y_test_pred_tuned, average='weighted')
recall_tuned = recall_score(y_test, y_test_pred_tuned, average='weighted')
f1_tuned = f1_score(y_test, y_test_pred_tuned, average='weighted')

print(f"Test Accuracy: {test_accuracy_tuned*100:.2f}%")
print(f"Precision: {precision_tuned*100:.2f}%")
print(f"Recall: {recall_tuned*100:.2f}%")
print(f"F1-Score: {f1_tuned*100:.2f}%")

print(f"\n Improvement over baseline: {(test_accuracy_tuned - test_accuracy)*100:.2f}%")

## 10. Save Final Model

In [ ]:
# Create complete pipeline with TF-IDF and model
final_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('classifier', best_tuned_model)
])

print(" Final Pipeline:")
print(final_pipeline)

In [ ]:
# Save model
model_filename = '../models/arabic_expense_classifier.joblib'
metadata_filename = '../models/arabic_model_metadata.json'

print(f"\n Saving model to: {model_filename}")
joblib.dump(final_pipeline, model_filename)
print(" Model saved!")

# Save metadata
metadata = {
    'model_name': best_model_name,
    'test_accuracy': float(test_accuracy_tuned),
    'test_precision': float(precision_tuned),
    'test_recall': float(recall_tuned),
    'test_f1': float(f1_tuned),
    'best_params': grid_search.best_params_,
    'categories': sorted(y.unique().tolist()),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
    'vocabulary_size': len(tfidf.vocabulary_),
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

print(f" Saving metadata to: {metadata_filename}")
with open(metadata_filename, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(" Metadata saved!")

In [ ]:
# Display metadata
print("\n Model Metadata:")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## 11. Test Predictions

In [ ]:
def predict_arabic_transaction(text):
    """
    Predict category for an Arabic transaction
    """
    # Preprocess
    processed = preprocess_arabic_text(text)
    
    # Predict
    prediction = final_pipeline.predict([processed])[0]
    probabilities = final_pipeline.predict_proba([processed])[0]
    confidence = max(probabilities)
    
    return {
        'original': text,
        'processed': processed,
        'predicted_category': prediction,
        'confidence': confidence
    }

# Test with sample transactions
test_transactions = [
    "دفع فاتورة الكهرباء",
    "شراء ملابس من المحل",
    "ركوب اوبر للمطار",
    "مرتب الشهر",
    "كشف عند الدكتور"
]

print("\n Testing Predictions:")
print("="*80)

for transaction in test_transactions:
    result = predict_arabic_transaction(transaction)
    print(f"\nOriginal: {result['original']}")
    print(f"Predicted: {result['predicted_category']}")
    print(f"Confidence: {result['confidence']*100:.2f}%")
    print("-"*80)

## 12. Pipeline Summary

In [ ]:
print("\n" + "="*80)
print(" PIPELINE SUMMARY")
print("="*80)

print(f"\n Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n Dataset:")
print(f"  - Total samples: {len(df):,}")
print(f"  - Training samples: {len(X_train):,}")
print(f"  - Test samples: {len(X_test):,}")
print(f"  - Categories: {y.nunique()}")

print("\n Best Model:")
print(f"  - Algorithm: {best_model_name}")
print(f"  - Best Parameters: {grid_search.best_params_}")

print("\n Performance Metrics:")
print(f"  - Test Accuracy: {test_accuracy_tuned*100:.2f}%")
print(f"  - Precision: {precision_tuned*100:.2f}%")
print(f"  - Recall: {recall_tuned*100:.2f}%")
print(f"  - F1-Score: {f1_tuned*100:.2f}%")

print("\n Saved Files:")
print(f"  - Model: {model_filename}")
print(f"  - Metadata: {metadata_filename}")

print("\n" + "="*80)
print(" PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)